In [ ]:
import mysql.connector
import time

# --- Configuration ---
db_config = {
    'host': 'localhost',
    'user': 'root',         # Replace with your MySQL username
    'password': '123', # Replace with your MySQL password
    'database': 'employees',
    'port': 8889
}

In [2]:
# --- Helper Functions ---
def measure_query(cursor, query, test_name):
    """Executes a query and measures the time it takes to fetch all results."""
    print(f"Running: {test_name}...")
    
    # Start high-resolution timer
    start_time = time.time()
    
    cursor.execute(query)
    results = cursor.fetchall() 
    
    end_time = time.time()
    duration = end_time - start_time
    
    print(f" -> Found {len(results)} rows.")
    print(f" -> Execution Time: {duration:.4f} seconds\n")
    return duration


In [3]:
# --- Helper Functions ---
def manage_index(cursor, action, index_name, table, columns=""):
    """Helper to create or drop indexes cleanly."""
    try:
        if action == "CREATE":
            print(f"Creating index '{index_name}' on {table}({columns})... This might take a moment.")
            cursor.execute(f"CREATE INDEX {index_name} ON {table}({columns})")
        elif action == "DROP":
            cursor.execute(f"DROP INDEX {index_name} ON {table}")
    except mysql.connector.Error as err:
        # Ignore errors if we try to drop an index that doesn't exist
        if action != "DROP": 
            print(f"Index Error: {err}")



In [8]:
# --- Main Test Suite ---
def run_tests():
    try:
        # Connect to Database
        conn = mysql.connector.connect(**db_config)
        cursor = conn.cursor()
        print("Connected to MySQL successfully!\n" + "="*40 + "\n")

        # ---------------------------------------------------------
        # TEST 1: Baseline (Full Table Scan vs Single Index)
        # ---------------------------------------------------------
        print("TEST 1: SINGLE COLUMN INDEX (last_name)")
        query_1 = "SELECT SQL_NO_CACHE * FROM employees WHERE last_name = 'Facello'"
        
        manage_index(cursor, "DROP", "idx_last_name", "employees") # Ensure clean slate
        
        measure_query(cursor, query_1, "Without Index (Full Table Scan)")
        
        manage_index(cursor, "CREATE", "idx_last_name", "employees", "last_name")
        measure_query(cursor, query_1, "With Index (Index Seek)")


        # ---------------------------------------------------------
        # TEST 2: JOIN Performance (Indexing the Driver Table)
        # ---------------------------------------------------------
        print("="*40)
        print("TEST 2: JOIN PERFORMANCE (hire_date filter)")
        query_2 = """
            SELECT SQL_NO_CACHE e.first_name, e.last_name, t.title 
            FROM employees e
            JOIN titles t ON e.emp_no = t.emp_no
            WHERE e.hire_date = '1989-09-12'
        """
        
        manage_index(cursor, "DROP", "idx_hire_date", "employees")
        
        measure_query(cursor, query_2, "JOIN Without hire_date Index (Scanning Driver Table)")
        
        manage_index(cursor, "CREATE", "idx_hire_date", "employees", "hire_date")
        measure_query(cursor, query_2, "JOIN With hire_date Index")


        # ---------------------------------------------------------
        # TEST 3: ORDER BY Performance (Avoiding Filesort)
        # ---------------------------------------------------------
        print("="*40)
        print("TEST 3: ORDER BY (Sorting Salaries)")
        query_3 = "SELECT SQL_NO_CACHE * FROM salaries ORDER BY salary DESC LIMIT 1000"
        
        manage_index(cursor, "DROP", "idx_salary", "salaries")
        
        measure_query(cursor, query_3, "ORDER BY Without Index (Filesort)")
        
        manage_index(cursor, "CREATE", "idx_salary", "salaries", "salary")
        measure_query(cursor, query_3, "ORDER BY With Index (Pre-sorted)")

        # --- Clean up indexes we made during the test ---
        print("="*40)
        print("Cleaning up created test indexes...")
        manage_index(cursor, "DROP", "idx_last_name", "employees")
        manage_index(cursor, "DROP", "idx_hire_date", "employees")
        manage_index(cursor, "DROP", "idx_salary", "salaries")
        print("Tests Complete!")

        # ---------------------------------------------------------
        # TEST 4: Wildcard vs prefix
        # ---------------------------------------------------------
        print("="*40)
        print("Test 4: LIKE SEARCH (Wildcard vs Prefix)")

        query_4a = "SELECT SQL_NO_CACHE * FROM employees WHERE last_name LIKE '%son'"

        query_4b = "SELECT SQL_NO_CACHE * FROM employees WHERE last_name LIKE '%Sch%'"

        manage_index(cursor, "DROP", 'idx_last_name2', "employees")
        measure_query(cursor, query_4a, "LIKE '%son' (Leading Wildcard - no Index)")

        manage_index(cursor, "CREATE", 'idx_last_name2', "employees", "last_name")
        measure_query(cursor, query_4b, "LIKE '%Sch' (Prefix - Uses Index)")

        # ---------------------------------------------------------
        # TEST 5: ORDER BY Performance (Avoiding Filesort)
        # ---------------------------------------------------------
        print("="*40)
        print("Test 5: Count vs Select * Performance")

        query_5a = "SELECT SQL_NO_CACHE * FROM salaries WHERE salary > 80000 LIMIT 1000"

        query_5b = "SELECT SQL_NO_CACHE COUNT(*) FROM salaries WHERE salary > 80000"

        measure_query(cursor, query_5a, "SELECT 8 WHERE salary > 80000")
        measure_query(cursor, query_5b, "COUNT(*) WHERE salary > 80000")

    except mysql.connector.Error as err:
        print(f"Error: {err}")
    finally:
        if 'conn' in locals() and conn.is_connected():
            cursor.close()
            conn.close()



In [9]:
if __name__ == "__main__":
    run_tests()

Connected to MySQL successfully!

TEST 1: SINGLE COLUMN INDEX (last_name)
Running: Without Index (Full Table Scan)...
 -> Found 186 rows.
 -> Execution Time: 0.0623 seconds

Creating index 'idx_last_name' on employees(last_name)... This might take a moment.
Running: With Index (Index Seek)...
 -> Found 186 rows.
 -> Execution Time: 0.0169 seconds

TEST 2: JOIN PERFORMANCE (hire_date filter)
Running: JOIN Without hire_date Index (Scanning Driver Table)...
 -> Found 121 rows.
 -> Execution Time: 0.4995 seconds

Creating index 'idx_hire_date' on employees(hire_date)... This might take a moment.
Running: JOIN With hire_date Index...
 -> Found 121 rows.
 -> Execution Time: 0.0132 seconds

TEST 3: ORDER BY (Sorting Salaries)
Running: ORDER BY Without Index (Filesort)...
 -> Found 1000 rows.
 -> Execution Time: 7.0297 seconds

Creating index 'idx_salary' on salaries(salary)... This might take a moment.
Running: ORDER BY With Index (Pre-sorted)...
 -> Found 1000 rows.
 -> Execution Time: 3.505